# Pilot status (CPU runtime is enough; no GPU units used)

Lists which pilots are complete and packs the results (text, JSON, metrics, summaries; no checkpoints)
into `frontier-llm-pilots-results.zip` on Drive. Download that zip and put it in the course repo under
`curriculum/pilots/incoming/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PILOT_DIR = '/content/drive/MyDrive/frontier-llm-pilots'
import os, glob, zipfile
ORDER = [('P1', 'A100', 'about 4.5 h (10m 0.3 h, 30m 1.3 h, 70m 3 h)'), ('P1B', 'A100', 'about 0.5 h'), ('P2', 'A100', 'about 0.5 h'), ('P9', 'A100', 'about 0.25 h'), ('P18', 'A100', 'minutes'), ('P3', 'A100', 'about 1.5 h'), ('P4', 'A100', 'about 1-2 h (needs P1 pilot-70m seed 0)'), ('P17', 'A100', 'about 1 h'), ('P13', 'A100', 'about 1 h'), ('P15', 'A100', 'about 1.5 h'), ('P20', 'A100', 'about 1-2 h'), ('P11', 'T4', 'about 2 h'), ('P16', 'T4', 'T4 variants'), ('P10', 'A100', 'about 2.5 h'), ('P5', 'A100', 'about 3 h'), ('P12', 'A100', 'about 3 h'), ('P14', 'A100', 'about 3 h'), ('P8', 'L4', 'about 3-4 h'), ('P7', 'A100', 'about 4 h'), ('P19', 'A100', 'about 4-8 h'), ('P6', 'A100', 'about 5 h')]
done = {os.path.basename(p)[:-9] for p in glob.glob(f'{PILOT_DIR}/.done/*-COMPLETE')}
for pid, gpu, hours in ORDER:
    n = len([p for p in glob.glob(f'{PILOT_DIR}/.done/{pid}-*') if not p.endswith('COMPLETE')])
    print(f"{pid:5} {'COMPLETE' if pid in done else ('partial, %d commands done' % n if n else 'not started'):28} {gpu:5} {hours}")
p1 = sorted(os.path.basename(os.path.dirname(f)) for f in glob.glob(f'{PILOT_DIR}/p1/*/metrics.jsonl'))
print('P1 runs with metrics:', p1)
z = f'{PILOT_DIR}/frontier-llm-pilots-results.zip'
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(PILOT_DIR):
        dirs[:] = [d for d in dirs if d not in ('data-v0-32k', 'data-v0-long', 'labdata')]
        for f in files:
            full = os.path.join(root, f)
            if f.endswith(('.txt', '.json', '.jsonl', '.yaml', '.md', '.csv')) and os.path.getsize(full) < 50e6:
                zf.write(full, os.path.relpath(full, PILOT_DIR))
print('wrote', z, round(os.path.getsize(z) / 1e6, 1), 'MB')